In [1]:
# CP2 Experiments — MAGIC Gamma Telescope Dataset

# Install additional libraries
!pip install xgboost lightgbm -q

import joblib
import pandas as pd

from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

from xgboost import XGBClassifier


RANDOM_STATE = 42


# Load dataset
df = pd.read_csv("magic04.data")

# Add column names
df.columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class",
]

print("Dataset shape:", df.shape)

# Encode target variable
encoder = LabelEncoder()
df["class"] = encoder.fit_transform(df["class"])

# Features and target
X = df.drop("class", axis=1)
y = df["class"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Models for comparison
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

results = []

best_model = None
best_score = 0

print("\nTraining models...\n")

# Train and evaluate models
for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append(
        {
            "Model": name,
            "Accuracy": round(accuracy, 4),
            "F1-score": round(f1, 4),
            "ROC-AUC": round(roc_auc, 4),
        }
    )

    if roc_auc > best_score:
        best_score = roc_auc
        best_model = model

# Hyperparameter tuning for Random Forest
print("\nRunning GridSearchCV...\n")

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=3,
    scoring="roc_auc",
)

grid_search.fit(X_train, y_train)

# Best tuned model
tuned_model = grid_search.best_estimator_

y_pred = tuned_model.predict(X_test)
y_proba = tuned_model.predict_proba(X_test)[:, 1]

tuned_accuracy = accuracy_score(y_test, y_pred)
tuned_f1 = f1_score(y_test, y_pred)
tuned_roc_auc = roc_auc_score(y_test, y_proba)

results.append(
    {
        "Model": "Random Forest tuned",
        "Accuracy": round(tuned_accuracy, 4),
        "F1-score": round(tuned_f1, 4),
        "ROC-AUC": round(tuned_roc_auc, 4),
    }
)

if tuned_roc_auc > best_score:
    best_score = tuned_roc_auc
    best_model = tuned_model

print("Best RF parameters:", grid_search.best_params_)
print("Best RF CV ROC-AUC:", round(grid_search.best_score_, 4))

# Results table
results_df = pd.DataFrame(results)

print("\nFinal Results:\n")
print(results_df.sort_values(by="ROC-AUC", ascending=False))

# Save best model
joblib.dump(best_model, "best_model.pkl")

print("\nBest model saved successfully.")

Dataset shape: (19019, 11)
Train shape: (15215, 10)
Test shape: (3804, 10)

Training models...

Training Logistic Regression...
Training Random Forest...
Training Gradient Boosting...
Training Extra Trees...
Training SVM...
Training XGBoost...

Running GridSearchCV...

Best RF parameters: {'max_depth': None, 'n_estimators': 200}
Best RF CV ROC-AUC: 0.9323

Final Results:

                 Model  Accuracy  F1-score  ROC-AUC
3          Extra Trees    0.8883    0.8276   0.9410
6  Random Forest tuned    0.8883    0.8306   0.9409
5              XGBoost    0.8885    0.8331   0.9406
1        Random Forest    0.8877    0.8304   0.9404
2    Gradient Boosting    0.8767    0.8082   0.9274
4                  SVM    0.8289    0.7098   0.8698
0  Logistic Regression    0.7981    0.6792   0.8446

Best model saved successfully.
